In [1]:
import os
os.environ['PATH'] += os.pathsep + '/Library/TeX/texbin'


import subprocess

try:
    subprocess.check_output(['latex', '--version'])
    print('LaTeX is now accessible from Python.')
except FileNotFoundError:
    print('LaTeX is still not accessible from Python.')

LaTeX is now accessible from Python.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('../../src/')  # Adjust the path as needed
from pca_full import pca_full

# Ensure reproducibility
np.random.seed(42)

# Set default text interpreter to LaTeX (for matplotlib)
plt.rcParams['text.usetex'] = True

# Define the OHspecial_transform function
def OHspecial_transform(df):
    names = df.columns.values
    data = df.values
    labels = []
    
    for j in range(len(names)):
        col_data = data[:, j]
        temp = np.unique(col_data[~np.isnan(col_data)])
        temp = temp[temp >= 0]
        
        if len(temp) == 0:
            # All data in column j is NaN or negative
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        elif len(temp) <= 2:
            # Keep original data
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        else:
            # One-hot encode
            var = np.zeros((data.shape[0], len(temp)))
            for l in range(len(temp)):
                var[:, l] = np.where(col_data == temp[l], 1, 0)
            idx = np.where(np.isnan(col_data))[0]
            var[idx, :] = np.nan
            # Use str instead of int to match MATLAB's num2str behavior
            label = [f"{names[j]}_{str(temp[l])}" for l in range(len(temp))]
        
        if j == 0:
            OH = var
        else:
            OH = np.hstack((OH, var))
        labels.extend(label)
    
    OH_df = pd.DataFrame(OH, columns=labels)
    return OH_df

# Read in data and remove non-cultural features + features missing > 50% of possible values

# From EA
KinOrgRaw = pd.read_csv('KinshipOrgDataRaw.csv')
SubRaw = pd.read_csv('SubsistenceDataRaw.csv')
# EA_All_Raw = pd.read_csv('EA_All.csv')  # Commented out in MATLAB code

# From Pulotu
IsoRaw = pd.read_csv('IsoDataRaw.csv')
ReligRaw = pd.read_csv('RelDataRaw.csv')

# Other datasets
Binford = pd.read_csv('Binford_all.csv')
Seshat = pd.read_csv('Seshat.csv')
Birds = pd.read_csv('data_birds_OH.csv')

# Apply OHspecial_transform to datasets
KinOrg = OHspecial_transform(KinOrgRaw)
Sub = OHspecial_transform(SubRaw)
Iso = OHspecial_transform(IsoRaw)
Relig = OHspecial_transform(ReligRaw)

for w in range(4, 5):

    if w == 1:
        X = KinOrg
    elif w == 2:
        X = Sub
    elif w == 3:
        X = Relig
    elif w == 4:
        X = Iso
    elif w == 5:
        X = Binford
    elif w == 6:
        X = Seshat
    elif w == 7:
        X = Birds

    # Remove features missing > 50% of their entries
    dim = X.shape
    X = X.iloc[:, 0:dim[1]]  # Redundant but kept for similarity
    cols = X.columns.values
    X = X.to_numpy()

    counts = []
    for j in range(dim[1]):
        temp = np.isnan(X[:, j]).astype(int)
        idx = np.where(temp == 1)[0]
        counts.append(len(idx))

    counts = np.array(counts) / dim[0]
    idx = np.where(counts > 0.5)[0]
    cols = np.delete(cols, idx)
    X = np.delete(X, idx, axis=1)
    dim = X.shape

    # Compute initial mean and standard deviation, ignoring NaNs
    MnInit = np.tile(np.nanmean(X, axis=0), (dim[0], 1))
    STDInit = np.tile(np.nanstd(X, axis=0), (dim[0], 1))

    if w < 5:
        X_ = X - MnInit
    elif w == 5:
        X_ = X
    elif w == 6:
        X_ = (X - MnInit) / STDInit
    elif w == 7:
        X_ = X

    X_ = X_.T

    MaxStop = min(dim)
    if MaxStop > 100:
        MaxStop = 100
    zz = np.arange(2, MaxStop + 1)

    # Initialize lists for dynamic data collection
    rms = []
    accu = []
    rmsA = []
    costA = []
    var_exp = []
    # X_start = pd.DataFrame(X_).to_numpy(copy=True)
    # X_start = np.copy(X_)
    # pd.DataFrame(X_).to_csv("python_X_before_for.csv", index=False)
    print(f"NaN count in X_ before for (", np.isnan(X_).sum())
    # print(f"NaN count in X_start before for (", np.isnan(X_start).sum())

    for y in range(MaxStop - 1):
        print (f"iteration {y} out of {MaxStop - 1}")
        # X_ = X_start
        # pd.DataFrame(X_).to_csv("python_X_after_for.csv", index=False)
        print(f"NaN count in X_ after for (iteration {y}):", np.isnan(X_).sum())
        # print(f"NaN count in X_start after for (iteration {y}):", np.isnan(X_start).sum())

        z = zz[y]
        print('###############################')
        print(f'Number of PCs: {z}')
        print('###############################')
        if y > 0:
            time.sleep(1)

        for k in range(1):  # Change if you want to do replications

            opts = {
                'maxiters': 80,
                'algorithm': 'vb',
                'uniquesv': 0,
                'rmsstop': [80, np.finfo(float).eps, np.finfo(float).eps],
                'cfstop': [80, 0, 0],
                'minangle': 0
            }

            result = pca_full(np.copy(X_), z, **opts)
            A = result['A']
            S = result['S']
            Mu = result['Mu']
            V = result['V']
            cv = result['cv']
            hp = result['hp']
            lc = result['lc']

            # print("*********** start ************")
            # print('A:')
            # print(A)
            # print('S:')
            # print(S)
            # print('Mu:')
            # print(Mu)
            # print('V:')
            # print(V)
            # print('cv:')
            # print(cv)
            # print('hp:')
            # print(hp)
            # print('lc:')
            # print(lc)
            # print("*********** end ************")

            if z == 20:
                plt.figure()
                plt.scatter(S[0, :], S[1, :])
                plt.title('Scatter plot of first two PCs')
                plt.show()

            print('Learning is finished.')

            Xrec = Mu + A @ S

            # Compute accuracy
            accu_num = np.abs(np.round(Xrec.T + MnInit) - X)
            accu_num = len(np.where(accu_num > 0)[0])
            accu_value = 1 - accu_num / len(np.where(~np.isnan(X_.flatten()))[0])
            accu.append(accu_value)

            # Record rms and append lc['rms'] and lc['cost']
            rms_value = lc['rms'][-1]
            rms.append(rms_value)
            rmsA.append(lc['rms'])
            costA.append(lc['cost'])

            # Compute explained variance
            vv = np.zeros(MaxStop)
            v = np.linalg.eigvals(np.cov(Xrec))
            v = np.flipud(np.sort(np.real(v)))
            v = v[:MaxStop]
            v = np.round(v, 4)
            v = v / np.sum(v)
            vv[:len(v)] = v
            var_exp.append(vv)

        print(f"Improvement: {rms[y] - rms[y - 1]}")
        if y > 0 and (rms[y] - rms[y - 1]) > 0:
            print("Breaking loop early due to positive improvement.")
            break

    # After the loop, convert lists to NumPy arrays
    rmsA = np.array(rmsA)
    costA = np.array(costA)
    var_exp = np.array(var_exp)
    rms = np.array(rms)
    accu = np.array(accu)

    # Since w == 7
    result = pca_full(X_, zz[y - 1], **opts)
    AFin = np.array(result['A'])
    SFin = np.array(result['S'])
    MuFin = np.array(result['Mu']).flatten()
    VFin = result['V']
    cvFin = result['cv']
    hpFin = result['hp']
    lcFin = result['lc']
    Xrec = MuFin[:, np.newaxis] + AFin @ SFin

    # Compute Vr
    Vr = np.zeros_like(X_)
    for i in range(X_.shape[0]):
        for j in range(X_.shape[1]):
            Vr[i, j] = (AFin[i, :] @ cvFin['S'][j] @ AFin[i, :].T +
                        SFin[:, j].T @ cvFin['A'][i] @ SFin[:, j] +
                        np.sum(cvFin['S'][j] * cvFin['A'][i]) +
                        cvFin['Mu'][i])
    VrFin = Vr

    # Save reconstructed data
    T = pd.DataFrame(Xrec.T, columns=cols)
    T.to_csv('Finch_VBPCA_Recon_All.csv', index=False)
    XrecFin = Xrec
    rmsFin = rms
    rmsAFin = rmsA
    costAFin = costA
    var_exp_Fin = var_exp
    accu_Fin = accu

    # Plot scatter of the first two PCs
    plt.figure()
    plt.scatter(SFin[0, :], SFin[1, :], c='b', marker='o')
    plt.title('Isolation')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.show()

    # Plot Accuracy and RMS vs Number of PCs
    plt.figure()
    fig, ax1 = plt.subplots()
    
    # Plot Accuracy on the left y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Number of PCs')
    ax1.set_ylabel('Accuracy', color=color)
    x_values = np.arange(2, len(accu_Fin) + 2)
    ax1.plot(x_values, accu_Fin, color=color, linewidth=2, marker='o')
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Plot RMS on the right y-axis
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('RMS', color=color)
    ax2.plot(x_values, rmsFin, color=color, linewidth=2, marker='o')
    ax2.tick_params(axis='y', labelcolor=color)
    
    # Add a vertical line at the final number of PCs
    ax1.axvline(x=len(accu_Fin) + 1, linewidth=2, color='black')
    
    # Set x-axis limits
    ax1.set_xlim([2, len(accu_Fin) + 1])
    
    # Add grid, title, and adjust font size
    ax1.grid(True)
    plt.title(r'Conflict \& Contact')
    plt.rcParams.update({'font.size': 18})
    
    # Show the plot
    plt.show()

    # Clear variables (optional in Python, but included for completeness)
    del rmsA
    del costA
    del var_exp
    del Vr
    del accu


ModuleNotFoundError: No module named 'src'